In [1]:
import matplotlib        as mpl
import matplotlib.pyplot as plt
import numpy             as np
import pandas            as pd

from scipy.optimize import minimize

In [30]:
tickers     = ['GOOG','CSCO','LOGI','AMZN','APPL']
est_rets    = {'GOOG':0.080,'CSCO':0.075,'LOGI':0.060,'AMZN':0.125,'APPL':0.100}
ann_std     = {'GOOG':0.287,'CSCO':0.363,'LOGI':0.462,'AMZN':0.340,'APPL':0.250}

corr = {('GOOG','GOOG'):1,   ('CSCO','CSCO'):1,   ('LOGI','LOGI'):1,   ('AMZN','AMZN'):1,    ('APPL','APPL'):1,
        ('GOOG','CSCO'):0.43,('GOOG','LOGI'):0.34,('GOOG','AMZN'):0.55,('GOOG','APPL'):0.62,
        ('CSCO','LOGI'):0.28,('CSCO','AMZN'):0.34,('CSCO','APPL'):0.43,
        ('LOGI','AMZN'):0.35,('LOGI','APPL'):0.39,
        ('AMZN','APPL'):0.58}

In [40]:
def sym_corr(tickers,corr):
    for t1 in tickers:
        for t2 in tickers:
            key = (t1,t2)
            if key in corr:
                pass
            else:
                key1 = (t2,t1)
                corr[key] = corr[key1]
    return corr

def create_cov(tickers,corr):    
    if len(corr) < len(tickers)**2:
        corr = sym_corr(tickers,corr)
    cov = {}
    for t1 in tickers:
        for t2 in tickers:
            cov[(t1,t2)] = ann_std[t1]*ann_std[t2]*corr[(t1,t2)]

    return cov

def calc_port_return(fs,tickers,est_rets):
    port_ret = 0
    ticker_idx  = {v: i for i, v in enumerate(tickers)}
    for t1 in tickers:
        port_ret += fs[ticker_idx[t1]]*est_rets[t1]
    return port_ret

def calc_port_std(fs,tickers,corr):
    ticker_idx  = {v: i for i, v in enumerate(tickers)}
    cov = create_cov(tickers,corr)
    port_var = 0
    for t1 in tickers:
        for t2 in tickers:
            port_var += fs[ticker_idx[t1]]*cov[(t1,t2)]*fs[ticker_idx[t2]]
    return np.sqrt(port_var)

def calc_port_metrics(fs,tickers,est_rets,cov):
    port_ret = calc_port_return(fs,tickers,est_rets)
    port_std = calc_port_std(fs,tickers,corr)
    return port_ret,port_std

In [55]:
fs = np.array([0.1,0.1,0.6,0.1,0.1])
print(calc_port_metrics(fs,tickers,est_rets,corr))
fs = np.array([0.12,0.1,0.0,0.19,0.59])
print(calc_port_metrics(fs,tickers,est_rets,corr))
fs = np.array([0.22,0.14,0.05,0.06,0.53])
print(calc_port_metrics(fs,tickers,est_rets,corr))
fs = np.array([0.0,0.0,0.0,1.0,0.0])
print(calc_port_metrics(fs,tickers,est_rets,corr))

(np.float64(0.07400000000000001), np.float64(0.3305669060871036))
(np.float64(0.09985), np.float64(0.23559933217222837))
(np.float64(0.0916), np.float64(0.23156752710170742))
(np.float64(0.125), np.float64(0.34))


In [56]:
np.sqrt(0.062)

np.float64(0.24899799195977465)

In [57]:
# your fixed parameters
target = 0.1

constraints = [
    {'type': 'eq',   'fun': lambda f: np.sum(f) - 1},
    {'type': 'ineq', 'fun': lambda f: calc_port_return(f, tickers, est_rets) - target}
]

bounds = [(0, 1)] * 5
x0 = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

result = minimize(
    calc_port_std,
    x0,
    args=(tickers, corr),   # <-- extra args passed here
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'ftol': 1e-9, 'maxiter': 1000, 'disp': True}
)

print(result.x)
print(result.fun)
print(calc_port_metrics(result.x,tickers,est_rets,corr))

Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.2357533346589545
            Iterations: 9
            Function evaluations: 54
            Gradient evaluations: 9
[0.11542788 0.09875251 0.         0.19109481 0.5947248 ]
0.2357533346589545
(np.float64(0.1000000000001244), np.float64(0.2357533346589545))


In [50]:
cov = create_cov(tickers,corr)

In [52]:
for k in cov:
    print(k,f'{cov[k]:0.3f}')

('GOOG', 'GOOG') 0.082
('GOOG', 'CSCO') 0.045
('GOOG', 'LOGI') 0.045
('GOOG', 'AMZN') 0.054
('GOOG', 'APPL') 0.044
('CSCO', 'GOOG') 0.045
('CSCO', 'CSCO') 0.132
('CSCO', 'LOGI') 0.047
('CSCO', 'AMZN') 0.042
('CSCO', 'APPL') 0.039
('LOGI', 'GOOG') 0.045
('LOGI', 'CSCO') 0.047
('LOGI', 'LOGI') 0.213
('LOGI', 'AMZN') 0.055
('LOGI', 'APPL') 0.045
('AMZN', 'GOOG') 0.054
('AMZN', 'CSCO') 0.042
('AMZN', 'LOGI') 0.055
('AMZN', 'AMZN') 0.116
('AMZN', 'APPL') 0.049
('APPL', 'GOOG') 0.044
('APPL', 'CSCO') 0.039
('APPL', 'LOGI') 0.045
('APPL', 'AMZN') 0.049
('APPL', 'APPL') 0.062
